In [0]:
# In this notebook we will train our geographic logistic regression models. 

# Fit separate logistic regression models for bike and dock stockout onset.
# The target is 1 if a new stockout occurs within the next 30 minutes, otherwise 0.
# Only currently non-stockout observations with sufficient future coverage are labeled.
#
# P(stockout) = 1 / (1 + exp(-eta))
#
# eta = intercept
#     + beta_1 * available
#     + beta_2 * available_fraction
#     + beta_3 * change_30min_fraction
#     + beta_4 * log_capacity
#     + beta_5 * weekend
#     + sum_k beta_k * harmonic_k
#     + geographic_intercept[cell]
#     + sum_k geographic_coefficient[cell, k] * harmonic_k
#
# available is the current number of rentable bikes or available docks.
# available_fraction is available / station capacity.
# change_30min_fraction is the recent change per minute divided by capacity.
# log_capacity is ln(capacity), with nonpositive or missing capacity set to null.
# weekend is 1 on Saturday and Sunday, otherwise 0.
#
# The four harmonics are sin(2*pi*h/24), cos(2*pi*h/24),
# sin(4*pi*h/24), and cos(4*pi*h/24), where h is fractional New York hour.
# Geographic cells have separate intercept adjustments and daily time curves.
# Every training cell has its own adjustments; unseen cells get none (the overall baseline).
#
# Numeric missing values are filled with training medians, then standardized.
# Coefficients minimize training log loss plus an L2 (ridge) penalty.
# The penalty acts like a normal(0, tau^2) prior on each coefficient, so cells with
# little data are shrunk toward the overall baseline (partial pooling).
# regParam = 1 / (n * tau^2), where n is the number of training rows, so the prior
# keeps its meaning as the training data grows.
# The prior scale tau is chosen by cross-validation, separately for bikes and docks.
#
# Each validation fold refits the full pipeline using only earlier data.
# Training labels must finish before validation begins.


import os

# Temporary storage used when saving and loading Spark models. 
# We need this because we are using serverless. technically we
# don't need the CREATE VOLUME IF NOT EXISTS
# But we must have the os.environ variable set so there is a
# place to put the results of our experiments at the end
# (Strictly, this is scratch space: Spark writes the model files here and MLflow
# copies them into the run. Metrics and other results go to the tracking server.)

spark.sql(
    "CREATE VOLUME IF NOT EXISTS citibike_project.citibike.mlflow_tmp"
)

os.environ["MLFLOW_DFS_TMP"] = (
    "/Volumes/citibike_project/citibike/mlflow_tmp"
)

from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
import math
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.spark
from mlflow import MlflowClient
from mlflow.models import infer_signature
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import (
    SQLTransformer, Imputer, StringIndexer, StringIndexerModel, OneHotEncoder,
    VectorAssembler, Interaction, StandardScaler,
)
from pyspark.ml.functions import vector_to_array

spark.conf.set("spark.sql.session.timeZone", "UTC")

# Job parameters. Scheduled jobs pass run_mode="production"; interactive runs
# default to "dev", which logs and registers models to separate locations.
dbutils.widgets.text("run_mode", "dev")
# Comma-separated prior scales (tau, log-odds units) to compare by CV.
# A scheduled job can pass one value, e.g. "0.25", to skip tuning.
dbutils.widgets.text("prior_scales", "0.0625,0.125,0.25,0.5,1.0")

RUN_MODE = dbutils.widgets.get("run_mode")
if RUN_MODE not in {"dev", "production"}:
    raise ValueError(f"Unknown run_mode: {RUN_MODE}")
IS_PRODUCTION = RUN_MODE == "production"
PRIOR_SCALES = sorted(float(s) for s in dbutils.widgets.get("prior_scales").split(","))

PREDICTOR_TABLE = "citibike_project.citibike.gold_station_predictors"
TARGET_TABLE = "citibike_project.citibike.gold_station_targets"
LOCAL_TZ = "America/New_York"
TRAIN_START = "2026-08-21"  # Local date, inclusive.
EVALUATION_END = None  # Exclusive local date; None uses the latest complete day.
MIN_TRAIN_DAYS = 30
VALIDATION_DAYS = 7
STEP_DAYS = 7
MAX_FOLDS = 3
HORIZON_MIN = 30
MAX_ITER = 200
TOLERANCE = 0.000001
# Save training runs in a shared MLflow experiment. Dev runs use their own.
EXPERIMENT_PATH = ("/Shared/citibike-model-training" if IS_PRODUCTION
                   else "/Shared/citibike-model-training-dev")

# Register the final fitted bike and dock models. Dev runs register to a separate
# schema, so the registration code is tested without touching production models.
MODEL_SCHEMA = ("citibike_project.citibike" if IS_PRODUCTION
                else "citibike_project.citibike_dev")
MODEL_NAMES = {
    "bike": f"{MODEL_SCHEMA}.bike_stockout",
    "dock": f"{MODEL_SCHEMA}.dock_stockout",
}
# Batch scoring loads models by alias, e.g. models:/<name>@champion.
CHAMPION_ALIAS = "champion"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {MODEL_SCHEMA}")
mlflow.set_experiment(EXPERIMENT_PATH)
mlflow.set_registry_uri("databricks-uc")

In [0]:

# Here we will create cross-validation folds. With great sadness and extreme irritation I couldn't find an
# elegant way to do the expanding window cross-validation, as is typical in time series modeling. Partly this is
# a deficiency of the spark ml libraries. There could be a variety of solutions, including training on a subsample
# using sklearn, or using mlforecast with a different "type" of model but this is just a demonstration model 
# so I'm a bit lazy to sort it out
#
# We will use an expanding window cross-validation
# scheme where we repeatedly train the model on old but time contiguous versions of the data and evalute the
# resulting model on the the next time period. In this case, we start with a minimum training days size
# of 30 days, and then add in 7 day windows. When I wrote this we were not comparing any models to each other,
# but the CV will still give us an estimate of our expected error as time changes.
# The CV now also chooses the prior scale: every candidate is scored on the same folds.

def local_midnight(day):
    return datetime.combine(day, datetime.min.time(), ZoneInfo(LOCAL_TZ))


def microseconds(moment):
    return int(moment.timestamp() * 1000000)


# I could not find a "library" method for doing this that would work in our case, much to my surprise
# This function isn't complicated its just that I like how the sklearn approach composes things
# so nicely.

# Returns a list of dictionaries, one per training/validation fold.
# Each dictionary contains:
#   fold: the fold number.
#   train_start: the fixed start of the expanding training period.
#   validation_start: where training ends and validation begins.
#   validation_end: where validation ends.
# All periods include their start and exclude their end.
# Only complete validation windows are included
# Returns date boundaries

def make_folds(start, end, minimum_days, validation_days, step_days, maximum_folds):
    if min(minimum_days, validation_days, step_days, maximum_folds) <= 0:
        raise ValueError("Window lengths and maximum folds must be positive.")
    cutoff = start + timedelta(days=minimum_days)
    folds = []
    while cutoff + timedelta(days=validation_days) <= end:
        folds.append({
            "fold": len(folds) + 1,
            "train_start": start,
            "validation_start": cutoff,
            "validation_end": cutoff + timedelta(days=validation_days),
        })
        cutoff += timedelta(days=step_days)
    return folds[-maximum_folds:]

# Read the latest version of our tables, and also calculate the start and end times

# Read fixed gold versions for this training run.
# Pinning versions means every re-read sees the same data (Spark re-reads the
# tables at each action) and lets us reproduce the run later from logged versions.
source_versions = {
    table: spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()["version"]
    for table in [PREDICTOR_TABLE, TARGET_TABLE]
}

predictors = (
    spark.read
    .option("versionAsOf", source_versions[PREDICTOR_TABLE])
    .table(PREDICTOR_TABLE)
)

# The target table has one row per station, poll time, and horizon.
# Keeping one horizon leaves one label per predictor row.
targets = (
    spark.read
    .option("versionAsOf", source_versions[TARGET_TABLE])
    .table(TARGET_TABLE)
    .filter(F.col("horizon_minutes") == HORIZON_MIN)
)

# Match predictors and targets at each station and poll time. 
data = predictors.join(
    targets.select(
        "station_id",
        "fetched_at",
        "label_end_microseconds",
        "bike_target",
        "dock_target",
    ),
    ["station_id", "fetched_at"],
    "inner",
)

harmonics = ["hour_sin_1", "hour_cos_1", "hour_sin_2", "hour_cos_2"]

# Exclude the latest partial local day.
# (.first() is the first point where Spark actually runs the join.)
latest_poll = data.select(
    F.max(F.col("fetched_at").cast("long")).alias("latest")
).first()["latest"]

if latest_poll is None:
    raise ValueError("No matching gold rows for the selected horizon.")

latest_date = datetime.fromtimestamp(
    latest_poll, ZoneInfo(LOCAL_TZ)
).date()

available_end = local_midnight(latest_date)
start = local_midnight(datetime.fromisoformat(TRAIN_START).date())
end = available_end

if EVALUATION_END:
    end = local_midnight(datetime.fromisoformat(EVALUATION_END).date())

if end > available_end:
    raise ValueError(
        f"Evaluation must end on or before {available_end.date()}."
    )

# Create the training and validation date windows.
folds = make_folds(
    start,
    end,
    MIN_TRAIN_DAYS,
    VALIDATION_DAYS,
    STEP_DAYS,
    MAX_FOLDS,
)



if not folds:
    required_end = start + timedelta(
        days=MIN_TRAIN_DAYS + VALIDATION_DAYS
    )
    raise ValueError(
        f"Need data through {required_end.date()} local midnight. "
        f"Available end: {end.date()}."
    )

display(pd.DataFrame(folds))

In [0]:
# We are defining some helper functions that select data, fit and evaluate models,
# and record training details in MLflow, a tool for tracking model runs and results.

numeric_inputs = [
    "available", "available_fraction", "change_30min_fraction",
    "log_capacity", "weekend", *harmonics,
]


# select_examples:
# Selects rows for the requested outcome ("bike" or "dock") in [begin, finish).
# Requires a known target and a target observation window ending before finish.
# Returns a DataFrame with the selected target copied to the numeric "label" column.
# (Conditions are Spark column expressions, hence & instead of "and".)

def select_examples(frame, outcome, begin, finish):
    return frame.filter(
        (F.col("fetched_at") >= F.lit(begin))
        & (F.col("fetched_at") < F.lit(finish))
        & (F.col("label_end_microseconds") < microseconds(finish))
        & F.col(f"{outcome}_target").isNotNull()
    ).withColumn("label", F.col(f"{outcome}_target").cast("double"))


# fit_model:
# Fits the geographic logistic regression pipeline using only the training rows.
# Creates common input names for bike or dock features and imputes training medians.
# Adds geographic indicators and their interactions with the four time harmonics.
# Every training cell gets its own indicator; unseen cells get zero adjustments.
# Sets the L2 penalty from the prior scale: regParam = 1 / (n * tau^2).
# Returns the fitted pipeline, the regParam used, and the number of training rows.

def fit_model(train, outcome, prior_scale):
    n = train.count()
    reg_param = 1.0 / (n * prior_scale ** 2)

    # Copy bike- or dock-specific columns to generic names. This is inside the
    # pipeline, so the saved model accepts raw gold-table columns when scoring.
    aliases = SQLTransformer(statement=f"""
        SELECT *,
            CAST(num_{outcome}s_available AS DOUBLE) AS available,
            {outcome}_available_fraction AS available_fraction,
            {outcome}_change_30min_per_minute_fraction AS change_30min_fraction
        FROM __THIS__
    """)
    imputed = [f"{column}_imputed" for column in numeric_inputs]
    imputer = Imputer(strategy="median", inputCols=numeric_inputs, outputCols=imputed)

    # Standardize numeric inputs ourselves (Spark scales vectors, so assemble first).
    # Cell indicators stay 0/1, so every cell gets the same prior.
    numeric_vector = VectorAssembler(inputCols=imputed, outputCol="numeric_raw")
    scaler = StandardScaler(inputCol="numeric_raw", outputCol="numeric_scaled",
                            withMean=True, withStd=True)

    # One-hot encoding in two Spark steps: cell name -> index -> indicator vector.
    # handleInvalid="keep" gives unseen cells an extra last slot, which the
    # encoder's default dropLast removes, so unseen cells get all zeros.
    indexer = StringIndexer(inputCol="geographic_cell", outputCol="geographic_index",
                            handleInvalid="keep")
    encoder = OneHotEncoder(inputCol="geographic_index", outputCol="geographic_indicators")

    # Each cell's own daily curve: every indicator times every harmonic.
    daily_curve = VectorAssembler(
        inputCols=[f"{column}_imputed" for column in harmonics], outputCol="daily_curve")
    cell_curves = Interaction(
        inputCols=["geographic_indicators", "daily_curve"], outputCol="geographic_daily_curves")

    # Spark models read a single vector column holding every input.
    features = VectorAssembler(
        inputCols=["numeric_scaled", "geographic_indicators", "geographic_daily_curves"],
        outputCol="features")

    # standardization=False keeps the penalty in the units set above.
    # MLflow's Spark flavor returns only the "prediction" column through its
    # generic interface. Writing probabilities there makes the registered model
    # return [P(no stockout), P(stockout)] instead of 0/1 class labels.
    model = LogisticRegression(
        maxIter=MAX_ITER, regParam=reg_param, elasticNetParam=0.0,
        standardization=False, tol=TOLERANCE,
        probabilityCol="prediction", predictionCol="")

    pipeline = Pipeline(stages=[
        aliases, imputer, numeric_vector, scaler, indexer, encoder,
        daily_curve, cell_curves, features, model,
    ])
    return pipeline.fit(train), reg_param, n


# evaluate_model:
# Applies the fitted pipeline to validation rows without refitting any stages.
# Returns a dictionary containing example and event counts, observed event rate,
# mean predicted probability, log loss, ROC AUC, and mean signed residual.
# Residual = observed minus predicted: positive means average underprediction.
# Clips probabilities only for log loss; AUC is undefined if only one class is present.

def evaluate_model(model, validation):
    # "prediction" holds [P(no stockout), P(stockout)]; [1] takes P(stockout).
    predictions = model.transform(validation).select(
        "label", "rawPrediction", vector_to_array("prediction")[1].alias("p"),
    )
    # Clip only for evaluating logarithms.
    p = F.greatest(F.lit(1e-12), F.least(F.lit(1 - 1e-12), F.col("p")))
    summary = predictions.agg(
        F.count("*").alias("examples"),
        F.sum("label").alias("events"),
        F.avg("label").alias("event_rate"),
        F.avg("p").alias("mean_probability"),
        F.avg(-F.col("label") * F.log(p) - (1 - F.col("label")) * F.log(1 - p)).alias("log_loss"),
    ).first().asDict()
    if not summary["examples"]:
        raise ValueError("No eligible validation examples in this fold.")
    summary["roc_auc"] = (
        BinaryClassificationEvaluator(metricName="areaUnderROC").evaluate(predictions)
        if 0 < summary["events"] < summary["examples"] else float("nan")
    )
    summary["mean_residual"] = summary["event_rate"] - summary["mean_probability"]
    return summary

# log_fit:
# Records the prior scale, regParam, cell count, feature count, training row count,
# and optimizer iterations in the active MLflow run.
# Saves the training objective history and prints a notice if MAX_ITER was reached.
# Does not save the fitted model or return a value.

def log_fit(model, prior_scale, reg_param, n):
    fitted = model.stages[-1]
    # The fitted cell list lives in the pipeline's StringIndexerModel stage.
    indexer = next(s for s in model.stages if isinstance(s, StringIndexerModel))
    cells = len(indexer.labels)
    mlflow.log_params({"prior_scale": prior_scale, "reg_param": reg_param,
                       "geographic_cells": cells, "fitted_features": fitted.numFeatures})
    mlflow.log_metrics({"iterations": fitted.summary.totalIterations,
                        "training_examples": n})
    # Expected: the numeric inputs, plus one indicator and four curve terms per cell.
    if fitted.numFeatures != len(numeric_inputs) + 5 * cells:
        print(f"Unexpected feature count {fitted.numFeatures}; check the cell encoding.")
    if fitted.summary.totalIterations >= MAX_ITER:
        print("Iteration limit reached; inspect the objective history.")
    mlflow.log_dict({"objective": [float(x) for x in fitted.summary.objectiveHistory]}, "objective_history.json")

In [0]:
# Fit the model to each training fold
# We name the model and save the results of each fit on each fold
# Each prior scale is fitted on every fold, as a nested run under one CV run.
# Serverless can't cache data, so each fit re-reads the pinned gold tables.


score_rows = []
run_name = "geographic-lr-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
with mlflow.start_run(run_name=run_name) as cv_run:
    cv_run_id = cv_run.info.run_id
    mlflow.set_tag("run_mode", RUN_MODE)
    mlflow.log_params({
        "train_start": TRAIN_START, "evaluation_end": end.isoformat(),
        "min_train_days": MIN_TRAIN_DAYS, "validation_days": VALIDATION_DAYS,
        "step_days": STEP_DAYS, "folds": len(folds), "horizon_minutes": HORIZON_MIN,
        "prior_scales": ",".join(str(s) for s in PRIOR_SCALES),
        "max_iter": MAX_ITER, "tolerance": TOLERANCE,
        "window_type": "expanding",
    })
    mlflow.log_dict(source_versions, "gold_versions.json")
    for prior_scale in PRIOR_SCALES:
        for fold in folds:
            for outcome in ["bike", "dock"]:
                print(f"tau {prior_scale}, fold {fold['fold']}: {outcome}, "
                      f"validation starts {fold['validation_start'].date()}")
                train = select_examples(data, outcome, fold["train_start"], fold["validation_start"])
                validation = select_examples(data, outcome, fold["validation_start"], fold["validation_end"])
                with mlflow.start_run(run_name=f"{outcome}-fold-{fold['fold']}-tau-{prior_scale}",
                                      nested=True) as run:
                    # MLflow params must be strings, so dates are converted.
                    mlflow.log_params({"outcome": outcome, **{
                        key: value.isoformat() if isinstance(value, datetime) else value
                        for key, value in fold.items()
                    }})
                    model, reg_param, n = fit_model(train, outcome, prior_scale)
                    metrics = evaluate_model(model, validation)
                    log_fit(model, prior_scale, reg_param, n)
                    # Skip NaN metrics (e.g. AUC for a one-class fold).
                    mlflow.log_metrics({key: float(value) for key, value in metrics.items() if math.isfinite(value)})
                    score_rows.append({
                        "outcome": outcome, "prior_scale": prior_scale, "fold": fold["fold"],
                        "validation_start": fold["validation_start"].date().isoformat(),
                        "validation_end": fold["validation_end"].date().isoformat(),
                        "run_id": run.info.run_id, **metrics,
                    })
    cv_scores = pd.DataFrame(score_rows)
    mlflow.log_text(cv_scores.to_csv(index=False), "fold_metrics.csv")

    # Combine folds per outcome and prior scale. Log loss is weighted by
    # validation examples, so larger folds count more.
    summary_rows = []
    for (outcome, prior_scale), group in cv_scores.groupby(["outcome", "prior_scale"]):
        summary_rows.append({
            "outcome": outcome, "prior_scale": prior_scale, "folds": len(group),
            "weighted_log_loss": (group.log_loss * group.examples).sum() / group.examples.sum(),
            "mean_fold_roc_auc": group.roc_auc.mean(),
            "worst_fold_log_loss": group.log_loss.max(),
        })
    cv_summary = pd.DataFrame(summary_rows)
    mlflow.log_text(cv_summary.to_csv(index=False), "cv_summary.csv")

    # Choose the prior scale with the lowest weighted log loss, per outcome.
    best_rows = cv_summary.loc[cv_summary.groupby("outcome").weighted_log_loss.idxmin()]
    best_prior_scale = dict(zip(best_rows.outcome, best_rows.prior_scale))
    mlflow.log_params({f"chosen_prior_scale_{outcome}": scale
                       for outcome, scale in best_prior_scale.items()})

display(cv_scores)
display(cv_summary)
print("Chosen prior scale:", best_prior_scale)
# A winner at the edge of the grid means the best value may lie beyond it.
if len(PRIOR_SCALES) > 1:
    for outcome, scale in best_prior_scale.items():
        if scale in (PRIOR_SCALES[0], PRIOR_SCALES[-1]):
            print(f"{outcome}: prior scale {scale} is at the edge of the grid; consider extending it.")

In [0]:
# We train a final model on all the data. This is what we will use to serve predictions
# Predictions come from a scheduled batch scoring job that loads the champion alias.
# (Databricks Model Serving can't host Spark ML models: its containers don't include Spark.)
# Each outcome uses the prior scale chosen by cross-validation.
# Every run registers a new version and moves the champion alias to it. This is a
# simplification: a stricter setup would first compare against the current champion.

client = MlflowClient(registry_uri="databricks-uc")
final_model_rows = []
for outcome in ["bike", "dock"]:
    prior_scale = best_prior_scale[outcome]
    train = select_examples(data, outcome, start, end)
    with mlflow.start_run(run_name=f"{outcome}-final-{run_name}") as run:
        mlflow.set_tags({"cv_run_id": cv_run_id, "run_mode": RUN_MODE})
        mlflow.log_params({"outcome": outcome, "train_start": start.isoformat(),
                           "train_end": end.isoformat(), "horizon_minutes": HORIZON_MIN,
                           "max_iter": MAX_ITER, "tolerance": TOLERANCE})
        mlflow.log_dict(source_versions, "gold_versions.json")
        model, reg_param, n = fit_model(train, outcome, prior_scale)
        log_fit(model, prior_scale, reg_param, n)
        # Raw gold-table columns the pipeline needs at scoring time.
        input_columns = ["geographic_cell", "log_capacity", "weekend", *harmonics,
                         f"num_{outcome}s_available", f"{outcome}_available_fraction",
                         f"{outcome}_change_30min_per_minute_fraction"]
        # The signature records input and output columns and types, built from the
        # model's real output: an array [P(no stockout), P(stockout)].
        example = train.select(*input_columns).limit(5)
        signature = infer_signature(example, model.transform(example).select("prediction"))
        saved = mlflow.spark.log_model(model, artifact_path="model", signature=signature)
        registered = mlflow.register_model(saved.model_uri, MODEL_NAMES[outcome])
        client.set_registered_model_alias(MODEL_NAMES[outcome], CHAMPION_ALIAS, registered.version)
        final_model_rows.append({"outcome": outcome, "prior_scale": prior_scale,
                                 "run_id": run.info.run_id, "model_uri": saved.model_uri,
                                 "model_name": MODEL_NAMES[outcome],
                                 "registered_version": registered.version})

display(pd.DataFrame(final_model_rows))